# 03 — Remplacez la Q-table par un cerveau neuronal : le DQN

Dans cet exercice, on va implémenter un **Deep Q-Network (DQN)** de deux façons :
1. **À la main avec PyTorch** — pour comprendre chaque rouage
2. **Avec Stable-Baselines3** — comme en production

Environnement cible : **CartPole-v1**

## 1. Pourquoi passer de la Q-table au DQN ?

Dans l'exercice précédent, on a utilisé une Q-table pour FrozenLake. Ça fonctionnait parce que l'environnement avait **16 états** — on pouvait stocker une valeur pour chaque paire (état, action) dans un tableau.

### Le problème des espaces continus

CartPole a un espace d'observation **continu** : 4 nombres réels (position, vitesse, angle, vitesse angulaire). Le nombre d'états possibles est **infini** — on ne peut pas faire de tableau.

| | FrozenLake | CartPole |
|--|-----------|----------|
| Espace d'états | Discret (16 états) | Continu (infini) |
| Q-table possible ? | ✅ Oui | ❌ Non |
| Solution | Q-table (16×4) | Réseau de neurones |

### La solution : approximer Q avec un réseau de neurones

L'idée du **DQN** (Mnih et al., DeepMind 2013) : remplacer la Q-table par un **réseau de neurones** qui prend un état en entrée et prédit les valeurs Q pour toutes les actions en sortie.

```
Q-TABLE (discret)             DQN (continu)
──────────────────            ─────────────────────────────────
État (entier 0-15)  →  Q(s,a)   État (vecteur 4D)  →  [Q(s,0), Q(s,1)]
                                          ↑
                               Réseau de neurones
                            (3 couches, ReLU, ~64 neurones)
```

### Les 3 innovations clés du DQN

| Innovation | Problème résolu | Sans ça... |
|------------|-----------------|------------|
| **Experience Replay** | Corrélation temporelle des données | L'agent sur-apprend sur les transitions récentes |
| **Target Network** | Instabilité de la cible d'apprentissage | La cible bouge en même temps que le réseau → instable |
| **Epsilon-greedy** | Exploration vs exploitation | L'agent n'explore pas assez (déjà vu en Q-Learning) |

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import collections
import matplotlib.pyplot as plt

print(f"PyTorch version : {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé  : {device}")

---
## PARTIE 1 — DQN from scratch avec PyTorch
---

## Étape 1 — La classe DQN (réseau de neurones)

### Architecture

Le réseau prend un **état** (vecteur de 4 nombres pour CartPole) et prédit une **valeur Q pour chaque action** possible.

```
Entrée (4)  →  Couche 1 (128)  →  ReLU  →  Couche 2 (128)  →  ReLU  →  Sortie (2)

[pos, vel, angle, ang_vel]                               [Q(s, gauche), Q(s, droite)]
```

### Rôle de ReLU

**ReLU** (Rectified Linear Unit) est la fonction d'activation la plus utilisée en deep learning :
- `ReLU(x) = max(0, x)`
- Elle introduit de la **non-linéarité** : sans elle, empiler des couches linéaires équivaut à une seule couche linéaire
- Elle est simple, rapide et évite le problème du gradient qui disparaît

### Pourquoi hériter de `nn.Module` ?

`nn.Module` est la classe de base de PyTorch pour tous les réseaux. En héritant, on obtient gratuitement :
- La gestion automatique des paramètres (`.parameters()`)
- La sauvegarde/chargement (`.state_dict()`)
- Le mode train/eval (`.train()`, `.eval()`)

In [ ]:
class DQN(nn.Module):
    """
    Réseau de neurones qui approxime la fonction Q.
    Entrée : état de l'environnement (vecteur)
    Sortie : valeur Q pour chaque action possible
    """

    def __init__(self, n_observations, n_actions):
        """
        n_observations : taille de l'espace d'observation (4 pour CartPole)
        n_actions      : nombre d'actions possibles (2 pour CartPole)
        """
        super(DQN, self).__init__()

        # Couches entièrement connectées (fully connected / linear)
        # nn.Linear(in_features, out_features) : y = Wx + b
        self.layer1 = nn.Linear(n_observations, 128)   # 4   → 128
        self.layer2 = nn.Linear(128, 128)              # 128 → 128
        self.layer3 = nn.Linear(128, n_actions)        # 128 → 2 (une valeur Q par action)

    def forward(self, x):
        """
        Passage en avant (forward pass) : calcule les valeurs Q pour l'état x.
        PyTorch appelle cette méthode automatiquement quand on fait : net(x)
        """
        x = F.relu(self.layer1(x))  # couche 1 + activation ReLU
        x = F.relu(self.layer2(x))  # couche 2 + activation ReLU
        return self.layer3(x)       # couche 3 : pas d'activation (valeurs Q peuvent être négatives)


# --- Tester la classe DQN ---
env_test = gym.make("CartPole-v1")
n_obs = env_test.observation_space.shape[0]  # 4
n_act = env_test.action_space.n              # 2
env_test.close()

net_test = DQN(n_obs, n_act)
print("Architecture du DQN :")
print(net_test)

# Compter les paramètres
n_params = sum(p.numel() for p in net_test.parameters())
print(f"\nNombre de paramètres : {n_params:,}")

# Test avec une observation fictive
obs_fictive = torch.FloatTensor([[0.1, -0.2, 0.05, 0.3]])  # batch de 1 observation
q_values = net_test(obs_fictive)
print(f"\nTest forward pass :")
print(f"  Entrée  : {obs_fictive.numpy()}")
print(f"  Sortie  : {q_values.detach().numpy()}  (Q gauche, Q droite)")

## Étape 1 (suite) — La classe ReplayBuffer (Experience Replay)

### Pourquoi stocker les expériences passées ?

Problème : si on entraîne le réseau à chaque pas de temps sur la **transition courante uniquement**, les données sont très corrélées (t, t+1, t+2 se ressemblent). Le réseau sur-apprend sur la séquence récente et "oublie" ce qu'il a appris avant.

**Solution — Experience Replay** :
1. On stocke toutes les transitions `(état, action, récompense, nouvel état)` dans un **buffer** (mémoire circulaire)
2. À chaque mise à jour, on tire un **batch aléatoire** dans ce buffer
3. Le mélange brise les corrélations → apprentissage plus stable

### Pourquoi `collections.deque` ?

`deque(maxlen=N)` est une file double à taille fixe :
- Quand elle est pleine, les anciennes expériences sont **automatiquement supprimées**
- Insertion en `O(1)` (beaucoup plus rapide qu'une liste Python pour ça)
- `maxlen` = capacité du buffer (on garde les N expériences les plus récentes)

In [ ]:
class ReplayBuffer:
    """
    Mémoire circulaire qui stocke les transitions (s, a, r, s', done).
    Permet l'Experience Replay : entraîner sur des mini-batchs aléatoires.
    """

    def __init__(self, capacity):
        """
        capacity : nombre maximum de transitions stockées.
        Quand le buffer est plein, les plus anciennes sont écrasées.
        """
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """
        Ajoute une transition au buffer.
        state      : observation avant l'action
        action     : action choisie (0 ou 1)
        reward     : récompense reçue (+1.0 dans CartPole)
        next_state : nouvelle observation après l'action
        done       : True si l'épisode est terminé
        """
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """
        Tire un mini-batch aléatoire du buffer.
        Retourne 5 listes : states, actions, rewards, next_states, dones
        """
        batch = random.sample(self.buffer, batch_size)

        # Dézipper : transformer une liste de tuples en 5 tuples séparés
        states, actions, rewards, next_states, dones = zip(*batch)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        """Retourne le nombre de transitions actuellement stockées."""
        return len(self.buffer)


# --- Tester le ReplayBuffer ---
buffer_test = ReplayBuffer(capacity=1000)

# Remplir avec des transitions fictives
for i in range(50):
    state = np.random.rand(4).astype(np.float32)
    action = random.randint(0, 1)
    reward = 1.0
    next_state = np.random.rand(4).astype(np.float32)
    done = (i % 10 == 9)
    buffer_test.push(state, action, reward, next_state, done)

print(f"Taille du buffer : {len(buffer_test)} transitions")

states, actions, rewards, next_states, dones = buffer_test.sample(batch_size=5)
print(f"Batch de 5 transitions — premiers états :")
for i, s in enumerate(states):
    print(f"  [{i}] état={np.round(s, 3)}, action={actions[i]}, reward={rewards[i]}, done={dones[i]}")

## Étape 2 — La boucle d'entraînement DQN

### Le Target Network : pourquoi deux réseaux ?

Le problème fondamental du DQN : pour calculer la perte (loss), on a besoin d'une **cible** (target). Cette cible est calculée avec l'équation de Bellman :

$$\text{target} = r + \gamma \cdot \max_{a'} Q_{\text{target}}(s', a')$$

Si on utilise le **même réseau** pour calculer les Q-values actuelles ET la cible, la cible bouge à chaque mise à jour → instabilité (comme essayer d'attraper sa propre ombre).

**Solution** : maintenir **deux réseaux identiques** :

| Réseau | Rôle | Mis à jour |
|--------|------|------------|
| `policy_net` | Choisit les actions, calcule Q(s,a) | À chaque batch (gradient descent) |
| `target_net` | Calcule la cible Q(s', a') | Périodiquement (copie de policy_net) |

En gardant `target_net` fixe pendant N steps, la cible est stable → apprentissage stable.

### La fonction `optimize_model` : le cœur du DQN

```
1. Tirer un batch du ReplayBuffer
2. Calculer Q(s, a) avec policy_net         ← ce qu'on PRÉDIT
3. Calculer max Q(s', a') avec target_net    ← ce qu'on CIBLE
4. target = r + γ · max Q(s', a')            ← équation de Bellman
5. loss = MSE(Q prédit, target)              ← erreur
6. Backprop + mise à jour des poids          ← apprentissage
```

In [ ]:
# =====================================================================
# HYPERPARAMÈTRES
# =====================================================================
BATCH_SIZE = 128        # nombre de transitions tirées du buffer à chaque optimisation
GAMMA = 0.99            # facteur d'actualisation (récompenses futures)
EPSILON_START = 1.0     # exploration initiale (100%)
EPSILON_END = 0.05      # exploration minimale (5%)
EPSILON_DECAY = 0.995   # facteur de réduction d'epsilon par épisode
LR = 1e-4               # learning rate de l'optimiseur Adam
MEMORY_SIZE = 10_000    # capacité du ReplayBuffer
TARGET_UPDATE = 10      # fréquence de copie policy_net → target_net (en épisodes)
N_EPISODES = 500        # nombre d'épisodes d'entraînement

print("Hyperparamètres définis.")

In [ ]:
# =====================================================================
# INITIALISATION
# =====================================================================
env = gym.make("CartPole-v1")

n_obs = env.observation_space.shape[0]  # 4
n_act = env.action_space.n              # 2

# Deux réseaux identiques en architecture
policy_net = DQN(n_obs, n_act).to(device)   # réseau actif (apprend à chaque batch)
target_net = DQN(n_obs, n_act).to(device)   # réseau cible (mis à jour périodiquement)

# Initialiser target_net avec les mêmes poids que policy_net
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()  # target_net ne s'entraîne pas → mode évaluation (désactive dropout, batchnorm)

# Optimiseur Adam : met à jour les poids de policy_net via gradient descent
optimizer = optim.Adam(policy_net.parameters(), lr=LR)

# Mémoire de replay
memory = ReplayBuffer(MEMORY_SIZE)

print("Initialisation terminée.")
print(f"policy_net → entraîné à chaque batch de {BATCH_SIZE} transitions")
print(f"target_net → copié depuis policy_net tous les {TARGET_UPDATE} épisodes")

In [ ]:
# =====================================================================
# FONCTION D'OPTIMISATION (le cœur du DQN)
# =====================================================================

def optimize_model():
    """Une étape d'apprentissage : tire un batch et met à jour policy_net."""

    # Pas assez d'expériences → on attend d'avoir rempli le buffer
    if len(memory) < BATCH_SIZE:
        return

    # ── 1. Tirer un mini-batch aléatoire du ReplayBuffer ──────────────
    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    # Convertir en tenseurs PyTorch
    # FloatTensor  = nombres réels (états, récompenses)
    # LongTensor   = entiers (actions, pour indexation)
    # BoolTensor   = booléens (dones)
    state_batch      = torch.FloatTensor(np.array(states)).to(device)
    action_batch     = torch.LongTensor(actions).unsqueeze(1).to(device)  # shape: (BATCH, 1)
    reward_batch     = torch.FloatTensor(rewards).to(device)
    next_state_batch = torch.FloatTensor(np.array(next_states)).to(device)
    done_batch       = torch.BoolTensor(dones).to(device)

    # ── 2. Calculer Q(s, a) avec policy_net ───────────────────────────
    # policy_net(state_batch) → shape: (BATCH, n_actions) = (128, 2)
    # .gather(1, action_batch) → sélectionne la valeur Q de l'action effectivement prise
    # Résultat shape: (BATCH, 1)
    q_values = policy_net(state_batch).gather(1, action_batch)

    # ── 3. Calculer max Q(s', a') avec target_net ─────────────────────
    # On n'a pas besoin du gradient ici (on calcule seulement la cible)
    with torch.no_grad():
        # target_net(next_state_batch) → shape: (BATCH, 2)
        # .max(1)[0] → valeur Q maximale pour chaque état suivant, shape: (BATCH,)
        next_q_values = target_net(next_state_batch).max(1)[0]

    # ── 4. Calculer la cible (équation de Bellman) ────────────────────
    # Si done=True : l'état suivant n'existe pas → la valeur future est 0
    # Si done=False : target = r + γ * max Q(s', a')
    # next_q_values.masked_fill(done_batch, 0.0) → met à 0 les Q-values des états terminaux
    target_q_values = reward_batch + GAMMA * next_q_values.masked_fill(done_batch, 0.0)
    target_q_values = target_q_values.unsqueeze(1)  # shape: (BATCH, 1) pour matcher q_values

    # ── 5. Calculer la perte (loss) ───────────────────────────────────
    # Smooth L1 Loss (= Huber loss) : moins sensible aux outliers que MSE
    # Compare Q prédit (policy_net) vs Q cible (target_net + Bellman)
    loss = F.smooth_l1_loss(q_values, target_q_values)

    # ── 6. Rétropropagation et mise à jour des poids ──────────────────
    optimizer.zero_grad()   # remettre les gradients à 0 (PyTorch accumule par défaut)
    loss.backward()         # calculer les gradients via backprop
    # Gradient clipping : limiter la taille des gradients pour éviter les explosions
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()        # mettre à jour les poids de policy_net


print("Fonction optimize_model définie.")

In [ ]:
# =====================================================================
# BOUCLE D'ENTRAÎNEMENT PRINCIPALE
# =====================================================================

episode_rewards = []
epsilon = EPSILON_START

for episode in range(N_EPISODES):

    # ── Début de l'épisode ────────────────────────────────────────────
    obs, info = env.reset()
    state = np.array(obs, dtype=np.float32)
    total_reward = 0
    done = False

    while not done:
        # ── Sélection d'action (epsilon-greedy) ───────────────────────
        if random.random() < epsilon:
            # EXPLORER : action aléatoire
            action = env.action_space.sample()
        else:
            # EXPLOITER : meilleure action selon policy_net
            with torch.no_grad():  # pas besoin de gradient pour l'inférence
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)  # shape: (1, 4)
                q_values = policy_net(state_tensor)  # shape: (1, 2)
                action = q_values.argmax(dim=1).item()  # indice de la valeur Q maximale

        # ── Exécuter l'action ─────────────────────────────────────────
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = np.array(next_obs, dtype=np.float32)
        done = terminated or truncated

        # ── Stocker la transition dans le ReplayBuffer ────────────────
        memory.push(state, action, reward, next_state, done)

        # Passer à l'état suivant
        state = next_state
        total_reward += reward

        # ── Optimiser policy_net ──────────────────────────────────────
        optimize_model()

    # ── Fin de l'épisode ──────────────────────────────────────────────
    episode_rewards.append(total_reward)

    # Réduire epsilon
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)

    # Copier policy_net → target_net tous les TARGET_UPDATE épisodes
    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    # Afficher la progression tous les 50 épisodes
    if (episode + 1) % 50 == 0:
        mean_reward = np.mean(episode_rewards[-50:])
        print(f"Épisode {episode + 1:4d} | Reward moy (50 der.) : {mean_reward:6.1f} | epsilon : {epsilon:.3f}")

env.close()
print("\nEntraînement DQN terminé !")

In [ ]:
# =====================================================================
# COURBE D'APPRENTISSAGE
# =====================================================================

window = 20
smoothed = [np.mean(episode_rewards[max(0, i - window):i + 1]) for i in range(len(episode_rewards))]

plt.figure(figsize=(11, 4))
plt.plot(episode_rewards, alpha=0.3, color='steelblue', linewidth=0.8, label='Reward brut')
plt.plot(smoothed, color='steelblue', linewidth=2, label=f'Moyenne glissante ({window} ép.)')
plt.axhline(y=500, color='green', linestyle='--', alpha=0.5, label='Maximum (500)')
plt.xlabel("Épisode")
plt.ylabel("Récompense totale")
plt.title("Courbe d'apprentissage — DQN from scratch sur CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Reward moyen sur les 50 derniers épisodes : {np.mean(episode_rewards[-50:]):.1f}")

### Analyse de la courbe

Typiquement, on observe **3 phases** :

| Phase | Épisodes | Ce qui se passe |
|-------|----------|-----------------|
| **Exploration** | 0 → ~100 | epsilon élevé, agent explore aléatoirement, rewards faibles (~20-50) |
| **Transition** | ~100 → ~300 | epsilon diminue, le réseau commence à apprendre, rewards augmentent |
| **Exploitation** | ~300 → 500 | epsilon bas, l'agent exploite ses connaissances, rewards ~400-500 |

Si la courbe ne monte pas, essayer :
- Augmenter `N_EPISODES` à 1000
- Diminuer `EPSILON_DECAY` (ex: 0.99) pour garder plus longtemps l'exploration
- Augmenter `MEMORY_SIZE`

---
## PARTIE 2 — DQN avec Stable-Baselines3
---

On vient de coder ~150 lignes pour implémenter DQN. SB3 fait tout ça en **4 lignes**.

### Pourquoi utiliser SB3 en pratique ?

| | DQN from scratch | DQN avec SB3 |
|--|-------------------|---------------|
| Lignes de code | ~150 | ~4 |
| Bugs potentiels | Nombreux | Quasi nuls (code testé) |
| Optimisations | À implémenter | Incluses |
| Personnalisation | Totale | Partielle (configs) |
| But pédagogique | Comprendre les rouages | Aller vite |

Faire les deux, c'est le meilleur des deux mondes : **comprendre** (from scratch) + **produire** (SB3).

In [ ]:
from stable_baselines3 import DQN as DQN_SB3
from stable_baselines3.common.evaluation import evaluate_policy
import os

os.makedirs("logs", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Imports SB3 OK")

## Étape 3 — DQN SB3 : entraîner, évaluer, sauvegarder

### `total_timesteps` vs nombre d'épisodes

> **Important** : SB3 raisonne en **timesteps** (pas de temps), pas en épisodes.

- 1 timestep = 1 appel à `env.step(action)`
- CartPole dure en moyenne ~200 steps par épisode (si bien entraîné)
- 25 000 timesteps ≈ 125 épisodes environ
- Pour vraiment converger, 50 000 - 100 000 steps est recommandé

In [ ]:
env_sb3 = gym.make("CartPole-v1")

# Instancier le DQN SB3
# 'MlpPolicy' = réseau de neurones dense (Multi-Layer Perceptron)
# c'est le bon choix pour des observations vectorielles (pas des images)
model_sb3 = DQN_SB3(
    'MlpPolicy',
    env_sb3,
    verbose=1,
    tensorboard_log="./logs/",
    learning_rate=1e-4,
    buffer_size=10_000,      # ReplayBuffer
    batch_size=128,
    gamma=0.99,              # facteur d'actualisation
    exploration_initial_eps=1.0,  # epsilon initial
    exploration_final_eps=0.05,   # epsilon final
    exploration_fraction=0.5,     # fraction des steps consacrés à la décroissance d'epsilon
    target_update_interval=500,   # fréquence de mise à jour du target_net (en steps)
)

print("\nDQN SB3 instancié.")
print("Architecture du policy_net :")
print(model_sb3.policy)

In [ ]:
# Entraîner le modèle
# total_timesteps = nombre total d'interactions avec l'environnement
model_sb3.learn(total_timesteps=25_000, tb_log_name="DQN_CartPole")

print("\nEntraînement SB3 terminé !")

In [ ]:
# Évaluer le modèle sur 100 épisodes
# evaluate_policy fait tourner l'agent en mode exploitation pure (pas d'exploration)
mean_reward, std_reward = evaluate_policy(
    model_sb3,
    env_sb3,
    n_eval_episodes=100,
    deterministic=True  # toujours la meilleure action, pas d'epsilon-greedy
)

print(f"Résultat sur 100 épisodes d'évaluation :")
print(f"  Reward moyen : {mean_reward:.2f}")
print(f"  Écart-type   : {std_reward:.2f}")
print(f"  (maximum possible : 500.0)")

if mean_reward >= 450:
    print("  Excellent ! L'agent a résolu CartPole.")
elif mean_reward >= 200:
    print("  Bon résultat. Plus de timesteps pour atteindre 450+.")
else:
    print("  À améliorer — essayer total_timesteps=100_000.")

In [ ]:
# Sauvegarder le modèle (SB3 ajoute .zip automatiquement)
model_sb3.save("models/dqn_cartpole")
print("Modèle sauvegardé : models/dqn_cartpole.zip")

# Pour recharger plus tard :
# model_recharge = DQN_SB3.load("models/dqn_cartpole", env=env_sb3)

env_sb3.close()

## Résumé comparatif

### Q-Learning vs DQN

| | Q-Learning | DQN |
|--|-----------|-----|
| Stockage de Q | Tableau (Q-table) | Réseau de neurones |
| Espace d'états | Discret et fini ✅ | Continu ✅ |
| Experience Replay | ❌ | ✅ (ReplayBuffer) |
| Target Network | ❌ | ✅ (stabilité) |
| Complexité | Faible | Moyenne |
| Environnements | FrozenLake, Taxi... | CartPole, Atari... |

### DQN from scratch vs SB3

| | From scratch | SB3 |
|--|-------------|-----|
| Lignes de code | ~150 | ~5 |
| Compréhension | Maximale | Limitée |
| Fiabilité | Dépend de l'implémentation | Code de prod testé |
| Flexibilité | Totale | Paramétrable |
| Usage | Apprentissage | Recherche / production |

### Concepts clés retenus

| Concept | Définition |
|---------|------------|
| **DQN** | Q-Learning avec un réseau de neurones à la place de la Q-table |
| **Experience Replay** | Stocker les transitions passées et apprendre sur des batchs aléatoires |
| **Target Network** | Deuxième réseau figé pour stabiliser la cible d'apprentissage |
| **Bellman target** | `r + γ · max Q(s', a')` — la valeur qu'on essaie de prédire |
| **Huber loss** | Fonction de perte robuste aux outliers (≈ MSE pour petites erreurs) |
| **Gradient clipping** | Limiter la taille des gradients pour éviter les explosions numériques |